In [2]:
import pandas as pd
from glob import glob
import json
import os

# Mapeamento de Críticas - V1

In [ ]:
quadros = sorted(glob("./_ignore_data_censo/Metadata/quadros/*.json"))

all_crit = []

for q in quadros:
    data = json.load(open(q))
    for questions in data.get('Quesitos', []):
        for sub_question in questions["Perguntas"]:
            if "Criticas" in sub_question:
                for critica in sub_question["Criticas"]:
                    if "Condicao" in critica:
                        all_crit.append({
                            "quadro": os.path.basename(q),
                            "pergunta": sub_question.get("Titulo", ""),
                            "variavel": sub_question["CodigoVariavel"],
                            "id": critica["Id"],
                            "condicao": critica["Condicao"],
                            "tipo": critica["Tipo"],
                            "mensagem": critica["Mensagem"]
                        })

df_criticas = pd.DataFrame(all_crit)

In [12]:
df_criticas

,quadro,pergunta,variavel,id,condicao,tipo,mensagem
0,quadro01.json,Telefone Fixo,V01150101,personalizada,V01150000 == 2 && V01150101 == null && V011501...,erro,Registre pelo menos um número de telefone (fix...
1,quadro01.json,Qual email?,V01160101,email,V01160100 != 2,erro,O email fornecido não está no formato correto....
2,quadro01.json,,V01180100,personalizada,V01180100 == 2 && V01180101 != 2 && V01180102 ...,erro,Pelo menos uma opção deve ser preenchida.
3,quadro01.json,Área,V01170100,personalizada,V01170100 == 0 && V01170500 != 2,erro,Deve ser preenchida a área total do estabeleci...
4,quadro01.json,Área,V01170100,personalizada,VW01170300 > 10000,alerta,Área total maior que 10.000 hectares. confirme...
...,...,...,...,...,...,...,...
326,quadro39.json,Formação de novas pastagens,V45012600,personalizada,V45012600 > 0 && V04060000 == 0 && V04070000 =...,alerta,"Valor com formação de novas pastagens, sem áre..."
327,quadro39.json,Outras despesas,V45012800,personalizada,(V05120200 == 2 || V05120500 == 2) && V4501280...,alerta,O estabelecimento recebe assistência técnica p...
328,quadro39.json,Outras despesas,V45012800,personalizada,VW45013600 > 500000,alerta,Valor das despesas acima do esperado. Confirme...
329,quadro40.json,CPF do produtor,V51010100,personalizada,V51010100 == null && V51010101 != 2,erro,Deve ser preenchido o número do CPF ou a opção...


In [ ]:
df_criticas.to_csv("./data/criticas_senso_2017_V1.csv", index=False)

# Mapeamento de Críticas - V2

In [8]:
quadros = sorted(glob("./_ignore_data_censo/Metadata/quadros/*.json"))

all_crit = []

for q in quadros:
    data = json.load(open(q))
    quad_exhibition_cond = ''
    quadHasExhibitionCond = bool(data.get("CondicaoDeExibicao", False))

    # Salva a condição de exibição do quadro, se houver
    if quadHasExhibitionCond:
        quad_exhibition_cond = f'({data.get("CondicaoDeExibicao")}) && '

    for questions in data.get('Quesitos', []):
        quest_exhibition_cond = ''
        questHasExhibitionCond = bool(questions.get("CondicaoDeExibicao", False))

        # Adiciona a condição de exibição do quesito, se houver
        if questHasExhibitionCond:
            quest_exhibition_cond = f'({questions.get("CondicaoDeExibicao")}) && '
        
        for sub_question in questions["Perguntas"]:
            sub_quest_exhibition_cond = ''
            subHasExhibitionCond = bool(sub_question.get("CondicaoDeExibicao", False))

            # Adiciona a condição de exibição da sub-pergunta, se houver
            if subHasExhibitionCond:
                sub_quest_exhibition_cond = f'({sub_question.get("CondicaoDeExibicao")}) && '
            
            if "Criticas" in sub_question:
                for critica in sub_question["Criticas"]:
                    if "Condicao" in critica:
                        all_crit.append({
                            "quadro": os.path.basename(q),
                            "pergunta": sub_question.get("Titulo", ""),
                            "variavel": sub_question["CodigoVariavel"],
                            "id": critica["Id"],
                            "condicao": critica["Condicao"],
                            "condicao_completa": f'{quad_exhibition_cond}{quest_exhibition_cond}{sub_quest_exhibition_cond}{critica["Condicao"]}',
                            "tipo": critica["Tipo"],
                            "mensagem": critica["Mensagem"]
                        })

df_criticas = pd.DataFrame(all_crit)
df_criticas

,quadro,pergunta,variavel,id,condicao,condicao_completa,tipo,mensagem
0,quadro01.json,Telefone Fixo,V01150101,personalizada,V01150000 == 2 && V01150101 == null && V011501...,(V01150000 == 2) && V01150000 == 2 && V0115010...,erro,Registre pelo menos um número de telefone (fix...
1,quadro01.json,Qual email?,V01160101,email,V01160100 != 2,(V01160100 == 2) && V01160100 != 2,erro,O email fornecido não está no formato correto....
2,quadro01.json,,V01180100,personalizada,V01180100 == 2 && V01180101 != 2 && V01180102 ...,V01180100 == 2 && V01180101 != 2 && V01180102 ...,erro,Pelo menos uma opção deve ser preenchida.
3,quadro01.json,Área,V01170100,personalizada,V01170100 == 0 && V01170500 != 2,V01170100 == 0 && V01170500 != 2,erro,Deve ser preenchida a área total do estabeleci...
4,quadro01.json,Área,V01170100,personalizada,VW01170300 > 10000,VW01170300 > 10000,alerta,Área total maior que 10.000 hectares. confirme...
...,...,...,...,...,...,...,...,...
326,quadro39.json,Formação de novas pastagens,V45012600,personalizada,V45012600 > 0 && V04060000 == 0 && V04070000 =...,V45012600 > 0 && V04060000 == 0 && V04070000 =...,alerta,"Valor com formação de novas pastagens, sem áre..."
327,quadro39.json,Outras despesas,V45012800,personalizada,(V05120200 == 2 || V05120500 == 2) && V4501280...,(V05120200 == 2 || V05120500 == 2) && V4501280...,alerta,O estabelecimento recebe assistência técnica p...
328,quadro39.json,Outras despesas,V45012800,personalizada,VW45013600 > 500000,VW45013600 > 500000,alerta,Valor das despesas acima do esperado. Confirme...
329,quadro40.json,CPF do produtor,V51010100,personalizada,V51010100 == null && V51010101 != 2,(V02010000 == 1 || V02010000 == 2 || V02010000...,erro,Deve ser preenchido o número do CPF ou a opção...


In [ ]:
df_criticas.to_csv("./data/criticas_senso_2017_V2.csv", index=False)